# Phase 11 — Denk-Tiefe vs. Switch: Budget-Forcing-Sweep + Entscheidungsbäume

Die Bestandsdaten (Phase 8b/8c) zeigen: Thinking beseitigt den Switch-Impuls nicht,
es **verlagert** ihn (16 % der Denk-Spuren des Tabellen-Patterns kippen selbst), und
die scheinbare Tiefen-Abhängigkeit ist durch Budget-Starvation + Pattern-Confound
unbestimmbar. Dieses Notebook misst die Übergangstiefe **kontrolliert**:

* dieselben Köder-Prompts über alle Tiefen (kein Pattern-Confound)
* Denk-Budget hart erzwungen: 0/256/512/1024/2048/4096 Tokens, Präfix wird bei
  Budget mit `</think>` geschlossen (kein Starvation-Bias — Antworten immer erzeugt)
* pro (Prompt × Tiefe): mehrere Denk-Präfixe × mehrere Antworten, resumable
* **Entscheidungsbaum am Antwortanfang** pro Tiefenstufe (exakte Wahrscheinlichkeiten)
* Stratifizierung: kippt die Antwort öfter, wenn das Denk-Präfix selbst gekippt war?

A100-Colab, Modell wie Phase 9/10. Ausgaben → Drive (`thinkdepth_*.jsonl`, JSON-Bäume).

In [ ]:
# === Cell 1 — config =========================================================
import os
try:
    from google.colab import userdata
    v=None
    try: v=userdata.get("HF_TOKEN")
    except Exception: v=None
    if v: os.environ.setdefault("HF_TOKEN",v)
except Exception as e:
    print("colab secrets unavailable:", e)
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF","expandable_segments:True")

MODEL    = os.environ.get("WEIRDSPEC_TARGET_MODEL","Qwen/Qwen3.6-35B-A3B-FP8")
DATA_DIR = "/content/drive/MyDrive/weirdspec"
OUT_PREF = "/content/drive/MyDrive/weirdspec/thinkdepth_prefixes.jsonl"
OUT_ANS  = "/content/drive/MyDrive/weirdspec/thinkdepth_answers.jsonl"
OUT_TREE = "/content/drive/MyDrive/weirdspec/thinkdepth_trees.json"

# the four decoy prompts (full prompt_ids; texts are loaded from weird_transcripts)
PROMPT_IDS = [
  "643fdf5dc86af81b9414db22a33a14ccb94582e2bfdee19816989ef5a280445fa5",  # Tabelle -> JP/KR (17/60 Kipper)
  "6427c18ab919f9d3dac61130206fa5105d6f3cee61fb7282b8d48ee26759f4f7a7",  # TensorFlow -> ZH (7/60)
  "64a66df34dec0e6bca3cd5260271ae1d38ac5380e8c701308acaeeb60226219075",  # Grid -> FR (13 Latein-Kipper)
  "645f57a93e74272bce86e835a201cd660a660ee17f44a06d71daffe3303c61a90f",  # Einkaufsliste -> FR
]
# Tiefen-Gitter v2 (nach Pilot): Aufloesung an den gemessenen Uebergaengen.
# CJK-Kipp stirbt bei <=512 -> feine flache Stufen; FR-Switch faellt erst
# zwischen 512 und 2048 -> mittlere Stufen. 4096 gestrichen (natuerliche
# Denk-Saettigung: Prompts schliessen ihr Denken selbst weit darunter).
DEPTHS_CJK = [0, 64, 128, 256, 512, 1024]
DEPTHS_FR  = [0, 512, 1024, 2048, 3072]
DEPTHS_BY_PID = {p: (DEPTHS_CJK if p.startswith(("643fdf5d","6427c18a")) else DEPTHS_FR)
                 for p in PROMPT_IDS}
DEPTHS = sorted(set(DEPTHS_CJK) | set(DEPTHS_FR))       # Union (fuer die Auswertung)
R_PREFIXES = 4        # Denk-Präfixe pro (Prompt x Tiefe) — gilt jetzt AUCH fuer d=0
                      # (4 leere Präfixe x 8 Antworten = 32er-Baseline statt 8)
ANS_PER_PREFIX = 8    # Antworten pro Präfix  -> 32 Antworten je Stufe
ANS_TOKENS = 24       # reicht zur Onset-Klassifikation
TREE_DEPTH = 6; TREE_PMIN = 0.02; TREE_CHILDREN = 4; TREE_NODES = 60
MOUNT_DRIVE = True
print("model:", MODEL, "| depths:", DEPTHS)

In [ ]:
# === Cell 2 — install + drive + prompts =====================================
import subprocess, sys, os, json, glob
subprocess.run([sys.executable,"-m","pip","install","-q","-U","transformers","accelerate"],check=True)
if MOUNT_DRIVE and not os.path.isdir(DATA_DIR):
    from google.colab import drive; drive.mount("/content/drive")
def read_jsonl(path):
    rows=[]
    with open(path,encoding="utf-8") as f:
        for line in f:
            line=line.strip()
            if line: rows.append(json.loads(line))
    return rows
def find_file(d,*names):
    for nm in names:
        p=os.path.join(d,nm)
        if os.path.isfile(p): return p
    for nm in names:
        h=sorted(glob.glob(os.path.join(d,"**",nm),recursive=True),key=len)
        if h: return h[0]
    return None
wp=find_file(DATA_DIR,"weird_transcripts.jsonl"); assert wp, "weird_transcripts.jsonl fehlt"
PROMPTS={}
for r in read_jsonl(wp):
    pid=r["id"].split("/")[0]
    if pid in PROMPT_IDS and pid not in PROMPTS:
        PROMPTS[pid]=next(t["content"] for t in r["conversations"] if t["role"]=="user")
assert len(PROMPTS)==len(PROMPT_IDS), f"nur {len(PROMPTS)}/{len(PROMPT_IDS)} Prompts gefunden"
for pid,tx in PROMPTS.items(): print(pid[:10],"->",tx[:90].replace("\n"," "))

In [ ]:
# === Cell 3 — model =========================================================
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
tokenizer=AutoTokenizer.from_pretrained(MODEL)
model=AutoModelForCausalLM.from_pretrained(MODEL,torch_dtype="auto",device_map="auto")
model.eval()
print("loaded:",model.config.model_type,"| device:",model.device)

In [ ]:
# === Cell 4 — pure logic (mock-testable, no torch) ==========================
import heapq, re
FOREIGN=[(0x0370,0x03FF),(0x0400,0x052F),(0x0530,0x058F),(0x0590,0x05FF),(0x0600,0x074F),
         (0x0900,0x097F),(0x0E00,0x0E7F),(0x3040,0x30FF),(0x3400,0x9FFF),(0xAC00,0xD7AF),(0xF900,0xFAFF)]
def script_run(t,run=3):
    c=0
    for ch in t:
        if ch.isalpha() and ord(ch)>=0x250 and any(a<=ord(ch)<=b for a,b in FOREIGN):
            c+=1
            if c>=run: return True
        elif ch.isalpha(): c=0
    return False
FR=set("le la les une un des est et pour avec dans votre vous voici bonjour du qui que sur cette ces aux ou par plus il elle nous sont".split())
EN=set("the is and for with in your you here of to that this are was were has have will would can it on as at be by".split())
def classify_answer(t):
    """takeover / gloss / latin-switch(fr) / english"""
    if not t.strip(): return "empty"
    al=[ch for ch in t if ch.isalpha()]
    fo=[ch for ch in al if ord(ch)>=0x250 and any(a<=ord(ch)<=b for a,b in FOREIGN)]
    if al and len(fo)/len(al)>=0.5: return "takeover"
    if script_run(t): return "gloss"
    w=re.findall(r"[a-zA-ZÀ-ſ']+",t.lower())
    fr=sum(1 for x in w if x in FR); en=sum(1 for x in w if x in EN)
    return "latin-switch(fr)" if (fr>=3 and fr>en) else "english"
def tok_script(s):
    for ch in s:
        o=ord(ch)
        if 0x3040<=o<=0x30FF: return "kana"
        if 0x3400<=o<=0x9FFF or 0xF900<=o<=0xFAFF: return "cjk"
        if 0xAC00<=o<=0xD7AF: return "hangul"
        if 0x0400<=o<=0x052F: return "cyrillic"
        if ch.isalpha() and o>=0x0250 and not (0x1E00<=o<=0x1EFF): return "other-nonlatin"
    if any(0x00C0<=ord(c)<=0x017F for c in s): return "latin-acc"
    return "latin"
def expand_tree(next_dist,prefix_ids,depth,p_min,max_children,max_nodes):
    """best-first token tree; next_dist(ids)->(probs desc, token_ids)"""
    root={"tok":None,"p":1.0,"path_p":1.0,"children":[],"_ids":list(prefix_ids),"d":0}
    heap=[(-1.0,0,root)]; n=0; uid=1
    while heap and n<max_nodes:
        _,_,node=heapq.heappop(heap)
        if node["d"]>=depth: continue
        probs,toks=next_dist(node["_ids"])
        for p,t in list(zip(probs,toks))[:max_children]:
            if p<p_min: break
            ch={"tok":int(t),"p":float(p),"path_p":node["path_p"]*float(p),
                "children":[],"_ids":node["_ids"]+[int(t)],"d":node["d"]+1}
            node["children"].append(ch); n+=1
            heapq.heappush(heap,(-ch["path_p"],uid,ch)); uid+=1
            if n>=max_nodes: break
    return root
def decorate(node,decode):
    if node["tok"] is not None:
        s=decode([node["tok"]]); node["text"]=s; node["script"]=tok_script(s)
    node.pop("_ids",None)
    for c in node["children"]: decorate(c,decode)
    return node
def print_tree(node,indent=0,min_p=0.02):
    if node.get("tok") is not None:
        print("  "*indent+"%r  p=%.3f  [%s]"%(node["text"].replace("\n","\\n"),node["p"],node["script"]))
    for c in sorted(node["children"],key=lambda c:-c["p"]):
        if c["p"]>=min_p: print_tree(c,indent+1,min_p)
def foreign_mass(node):
    """probability mass of non-latin branches among the root's children.
    NOTE: blind to the FRENCH switch by construction (latin script) - for the
    FR prompts the answer-level rates are the primary readout, not P_fremd."""
    ch=node["children"]
    return sum(c["p"] for c in ch if c.get("script") in ("kana","cjk","hangul","cyrillic","other-nonlatin"))
def think_prefix(user_text,think_text=None):
    base="<|im_start|>user\n"+user_text+"<|im_end|>\n<|im_start|>assistant\n<think>\n"
    if think_text is None: return base
    return base+think_text+"\n</think>\n\n"
def wilson(k,n,z=1.96):
    import math
    if n==0: return (0.0,0.0,0.0)
    p=k/n; d=1+z*z/n; c=p+z*z/(2*n)
    h=z*math.sqrt(p*(1-p)/n+z*z/(4*n*n))
    return p,(c-h)/d,(c+h)/d
def transition_window(depths,ks,ns):
    """(last depth with rate >= half of depth-0 rate, first depth below) or None"""
    if not ns or ns[0]==0 or ks[0]==0: return None
    base=ks[0]/ns[0]; last_hi=depths[0]; first_lo=None
    for d,k,n in zip(depths[1:],ks[1:],ns[1:]):
        if n==0: continue
        if k/n>=base/2: last_hi=d
        elif first_lo is None: first_lo=d
    return (last_hi,first_lo) if first_lo is not None else None
print("logic ready")

In [ ]:
# === Cell 5 — reasoning prefixes per (prompt x depth), resumable ============
import json, os, torch
END_THINK="</think>"
done=set()
if os.path.exists(OUT_PREF):
    for r in (json.loads(l) for l in open(OUT_PREF) if l.strip()):
        done.add((r["pid"],r["depth"],r["prefix_idx"]))
f=open(OUT_PREF,"a",encoding="utf-8")
@torch.no_grad()
def gen(prefix,max_new,temp=1.0):
    ids=tokenizer(prefix,return_tensors="pt").input_ids.to(model.device)
    out=model.generate(ids,do_sample=True,temperature=temp,top_p=1.0,top_k=0,
                       repetition_penalty=1.0,max_new_tokens=max_new,
                       pad_token_id=tokenizer.eos_token_id)
    return tokenizer.decode(out[0,ids.shape[1]:],skip_special_tokens=False)
n_done=0
for pid,user in PROMPTS.items():
    for depth in DEPTHS_BY_PID[pid]:
        for j in range(R_PREFIXES):    # auch d=0: 4 leere Präfixe -> 32 Antworten Baseline
            key=(pid,depth,j)
            if key in done: continue
            if depth==0:
                think=""
            else:
                raw=gen(think_prefix(user),depth)
                think=raw.split(END_THINK)[0]             # Modell schloss selbst frueher
            row=dict(pid=pid,depth=depth,prefix_idx=j,think=think,
                     think_chars=len(think),self_closed=(depth>0 and END_THINK in (raw if depth>0 else "")),
                     think_flipped=script_run(think))
            f.write(json.dumps(row,ensure_ascii=False)+"\n"); f.flush(); n_done+=1
            print("prefix %s d=%-5d #%d chars=%-6d flipped=%s"%(pid[:8],depth,j,len(think),row["think_flipped"]))
f.close(); print("new prefixes:",n_done)

In [ ]:
# === Cell 6 — answers per prefix, resumable =================================
import json, os, torch
prefixes=[json.loads(l) for l in open(OUT_PREF) if l.strip()]
_seen=set(); pfx=[]
for r in prefixes:
    k=(r["pid"],r["depth"],r["prefix_idx"])
    if k not in _seen: pfx.append(r); _seen.add(k)
done=set()
if os.path.exists(OUT_ANS):
    for r in (json.loads(l) for l in open(OUT_ANS) if l.strip()):
        done.add((r["pid"],r["depth"],r["prefix_idx"]))
f=open(OUT_ANS,"a",encoding="utf-8")
@torch.no_grad()
def gen_batch(prefix,n,max_new):
    ids=tokenizer(prefix,return_tensors="pt").input_ids.to(model.device)
    out=model.generate(ids,do_sample=True,temperature=1.0,top_p=1.0,top_k=0,
                       repetition_penalty=1.0,max_new_tokens=max_new,
                       num_return_sequences=n,pad_token_id=tokenizer.eos_token_id)
    return [tokenizer.decode(o[ids.shape[1]:],skip_special_tokens=True) for o in out]
for r in pfx:
    key=(r["pid"],r["depth"],r["prefix_idx"])
    if key in done: continue
    full=think_prefix(PROMPTS[r["pid"]],r["think"])
    answers=gen_batch(full,ANS_PER_PREFIX,ANS_TOKENS)
    f.write(json.dumps(dict(pid=r["pid"],depth=r["depth"],prefix_idx=r["prefix_idx"],
        think_flipped=r["think_flipped"],think_chars=r["think_chars"],
        answers=answers,cls=[classify_answer(a) for a in answers]),ensure_ascii=False)+"\n")
    f.flush()
    print("answers %s d=%-5d #%d -> %s"%(r["pid"][:8],r["depth"],r["prefix_idx"],
          [classify_answer(a) for a in answers]))
f.close(); print("answers done")

In [ ]:
# === Cell 7 — decision trees at the answer start, per (prompt x depth) ======
import json, torch, numpy as np
@torch.no_grad()
def next_dist(ids,topn=16):
    t=torch.tensor([ids],device=model.device)
    logits=model(t).logits[0,-1].float()
    p=torch.softmax(logits,-1)
    pr,ix=torch.topk(p,topn)
    return pr.cpu().numpy(),ix.cpu().numpy()
prefixes=[json.loads(l) for l in open(OUT_PREF) if l.strip()]
best={}   # representative prefix per (pid,depth): the first
for r in prefixes:
    best.setdefault((r["pid"],r["depth"]),r)
TREES={}
for (pid,depth),r in sorted(best.items()):
    full=think_prefix(PROMPTS[pid],r["think"])
    ids=tokenizer(full,return_tensors="pt").input_ids[0].tolist()
    tree=expand_tree(next_dist,ids,TREE_DEPTH,TREE_PMIN,TREE_CHILDREN,TREE_NODES)
    decorate(tree,lambda t:tokenizer.decode(t))
    TREES["%s|%d"%(pid,depth)]=dict(pid=pid,depth=depth,think_flipped=r["think_flipped"],
                                    foreign_mass=foreign_mass(tree),tree=tree)
    print("="*90)
    print("TREE %s  depth=%d  (P_fremd am Antwortstart=%.3f, think_flipped=%s)"
          %(pid[:10],depth,foreign_mass(tree),r["think_flipped"]))
    print_tree(tree)
json.dump(TREES,open(OUT_TREE,"w"),ensure_ascii=False)
print("\ntrees ->",OUT_TREE)

In [ ]:
# === Cell 8 — analysis: dose-response, transition window, stratification ====
import json, numpy as np, matplotlib.pyplot as plt, collections
ans=[json.loads(l) for l in open(OUT_ANS) if l.strip()]
_seen=set(); rows=[]
for r in ans:
    k=(r["pid"],r["depth"],r["prefix_idx"])
    if k not in _seen: rows.append(r); _seen.add(k)
SWITCH=("takeover","gloss","latin-switch(fr)")
print("SWITCH-RATE vs erzwungene Denk-Tiefe (Wilson 95%):")
fig,ax=plt.subplots(1,2,figsize=(12,4.2))
pref_by={}
try:
    for r in (json.loads(l) for l in open(OUT_PREF) if l.strip()):
        pref_by.setdefault((r["pid"],r["depth"]),[]).append(r["think_chars"])
except FileNotFoundError: pass
for pid in PROMPT_IDS:
    ds,ps,los,his=[],[],[],[]
    for d in DEPTHS_BY_PID.get(pid,DEPTHS):
        cl=[c for r in rows if r["pid"]==pid and r["depth"]==d for c in r["cls"]]
        if not cl: continue
        k=sum(1 for c in cl if c in SWITCH)
        p,lo,hi=wilson(k,len(cl))
        ds.append(d); ps.append(p); los.append(p-lo); his.append(hi-p)
        mc=pref_by.get((pid,d),[])
        sat=" SATURIERT (natuerl. Denklaenge erreicht)" if (d>0 and mc and
             sum(mc)/len(mc)<3.5*d) else ""
        print("  %s d=%-5d n=%-3d rate=%.1f%% [%.1f,%.1f]  %s%s"%(pid[:8],d,len(cl),100*p,
              100*(p-los[-1]),100*(p+his[-1]),dict(collections.Counter(cl)),sat))
    ax[0].errorbar([d+1 for d in ds],[100*p for p in ps],yerr=[[100*l for l in los],[100*h for h in his]],
                   marker="o",capsize=3,label=pid[:8])
    tw=transition_window(ds,[sum(1 for r in rows if r["pid"]==pid and r["depth"]==d
        for c in r["cls"] if c in SWITCH) for d in ds],
        [sum(len(r["cls"]) for r in rows if r["pid"]==pid and r["depth"]==d) for d in ds])
    # Hinweis: P_fremd der Baeume ist fuer FR-Prompts konstruktionsbedingt blind
    print("  -> Uebergangsfenster %s: %s"%(pid[:8],tw))
ax[0].set_xscale("symlog"); ax[0].set_xlabel("erzwungene Denk-Tiefe (Tokens, +1)")
ax[0].set_ylabel("Switch-Rate der Antwort (%)"); ax[0].legend(fontsize=8,frameon=False)
ax[0].set_title("Dosis-Wirkung: Denk-Tiefe vs. Switch")
# stratification: flipped vs clean thinking prefixes (depth>0 pooled)
lab=[]; vals=[]
for fl in (False,True):
    cl=[c for r in rows if r["depth"]>0 and r["think_flipped"]==fl for c in r["cls"]]
    k=sum(1 for c in cl if c in SWITCH); p,lo,hi=wilson(k,len(cl))
    lab.append("Denken sauber\n(n=%d)"%len(cl) if not fl else "Denken gekippt\n(n=%d)"%len(cl))
    vals.append((100*p,100*(p-lo),100*(hi-p)))
    print("think_flipped=%s: rate=%.1f%% [%.1f,%.1f] n=%d"%(fl,100*p,100*lo,100*hi,len(cl)))
ax[1].bar(range(2),[v[0] for v in vals],yerr=[[v[1] for v in vals],[v[2] for v in vals]],
          capsize=4,color=["#2563EB","#DC2626"],alpha=0.85)
ax[1].set_xticks(range(2)); ax[1].set_xticklabels(lab,fontsize=9)
ax[1].set_ylabel("Switch-Rate der Antwort (%)")
ax[1].set_title("Verlagerungs-These: kippt die Antwort oefter,\nwenn das Denken selbst gekippt war?")
# tree summary: P_foreign at answer start vs depth
try:
    TREES=json.load(open(OUT_TREE))
    print("\nP(fremdes Token) am Antwortstart (aus den Baeumen):")
    for key,t in sorted(TREES.items()):
        print("  %s d=%-5d P_fremd=%.3f"%(t["pid"][:8],t["depth"],t["foreign_mass"]))
except FileNotFoundError:
    print("(Baeume noch nicht erzeugt - Cell 7)")
plt.tight_layout(); plt.show()

## Cell 9 — Prompt-Perplexität: Wie künstlich sind die Köder wirklich?

Perplexität jedes Language-Switching-Prompts unter dem Subjektmodell, doppelt
verankert (16 natürliche Nutzer-Prompts inkl. drei Köder-Zwillingen + Katalog-
Prompts der anderen Behaviors), kombiniert mit Naturalness-Elo, match_rate und
Köder-Index in einer Tabelle. Läuft in der Phase-11-Session (Modell geladen).

In [ ]:
# === Cell 9 — Prompt-Perplexität: Köder vs. natürliche Prompts ==============
# Misst die Perplexität jedes Language-Switching-Prompts unter dem SUBJEKT-
# Modell (als User-Turn im Chat-Template) und verankert sie doppelt:
#   * 16 handgeschriebene natürliche Nutzer-Prompts (darunter drei "Zwillinge"
#     der Köder: echte TensorFlow-Frage, echte Einkaufsliste, echtes 10x10-Grid)
#   * die Prompts der übrigen Behaviors des Katalogs (bis 120, als Verteilung)
# Ausgabe: EINE kombinierte Tabelle mit Naturalness-Elo, P_nat, match_rate,
# Köder-Index und PPL-Ankern. Braucht model+tokenizer aus Cell 3 + Drive.
import json, math, os, glob, statistics, collections, torch, numpy as np
DATA_DIR=globals().get("DATA_DIR","/content/drive/MyDrive/weirdspec")
def _rj(p):
    rows=[]
    with open(p,encoding="utf-8") as f:
        for line in f:
            line=line.strip()
            if line: rows.append(json.loads(line))
    return rows
def _ff(d,*names):
    for nm in names:
        q=os.path.join(d,nm)
        if os.path.isfile(q): return q
    for nm in names:
        h=sorted(glob.glob(os.path.join(d,"**",nm),recursive=True),key=len)
        if h: return h[0]
    return None
NATURAL=[
 "Can you help me write a polite email to my landlord about a broken heater? It's been three days and the apartment is getting cold.",
 "What's the difference between a Roth IRA and a traditional IRA? I'm 28 and just started my first full-time job.",
 "My pasta sauce always turns out too acidic. What can I do to balance it?",
 "Explain how HTTPS works in simple terms.",
 "I have a 10x10 grid in Excel and want to highlight every cell above the column average. How do I do that with conditional formatting?",
 "Write a short bedtime story about a dragon who is afraid of the dark.",
 "What should I see in Lisbon in three days? We like food and museums, not so much nightlife.",
 "Can you review this sentence for grammar: 'Neither of the managers were available for comments.'",
 "How do I center a div horizontally and vertically in CSS?",
 "Summarize the plot of Hamlet in five sentences.",
 "My dog keeps pulling on the leash. Any training tips?",
 "What is TensorFlow and what is it used for?",
 "Please make me a grocery list for a week of vegetarian dinners for two people.",
 "Convert this table of monthly expenses into a budget summary: rent 1200, food 400, transport 90, subscriptions 35.",
 "Why is the sky blue during the day but red at sunset?",
 "Draft a LinkedIn post announcing that I'm starting a new role as a data analyst.",
]
HDR_IDS=tokenizer("<|im_start|>user\n").input_ids
@torch.no_grad()
def prompt_ppl(text):
    full="<|im_start|>user\n"+text+"<|im_end|>"
    ids=tokenizer(full,return_tensors="pt",truncation=True,max_length=768).input_ids.to(model.device)
    lp=model(ids).logits[0,:-1].float().log_softmax(-1)
    tgt=ids[0,1:]
    lps=lp[torch.arange(len(tgt)),tgt][len(HDR_IDS)-1:-1]   # nur Prompt-Inhalt, ohne im_end
    return float(torch.exp(-lps.mean())) if lps.numel() else float("nan")

mp=_ff(DATA_DIR,"weird_meta.jsonl"); wp=_ff(DATA_DIR,"weird_transcripts.jsonl")
meta=_rj(mp); id2m={r["transcript_id"]:r for r in meta}
LS="language-switching-english"
# repräsentativer Prompt je LS-Pattern = häufigste prompt_id
byp=collections.defaultdict(collections.Counter)
elo={}; mrate={}
for r in meta:
    if r["behavior_id"]==LS:
        byp[r["pattern_id"]][r["prompt_id"]]+=1
        elo[r["pattern_id"]]=r.get("elo_prompt_naturalness"); mrate[r["pattern_id"]]=r.get("match_rate")
rep={pat:c.most_common(1)[0][0] for pat,c in byp.items()}
need={pid for pid in rep.values()}
other_ids={}   # prompt_id -> (behavior) fuer Katalog-Referenz
for r in meta:
    if r["behavior_id"]!=LS and r["prompt_id"] not in other_ids:
        other_ids[r["prompt_id"]]=r["behavior_id"]
texts={}; other_texts={}
for r in _rj(wp):
    pid=r["id"].split("/")[0]
    if pid in need and pid not in texts:
        texts[pid]=next(t["content"] for t in r["conversations"] if t["role"]=="user")
    elif pid in other_ids and pid not in other_texts and len(other_texts)<120:
        other_texts[pid]=next(t["content"] for t in r["conversations"] if t["role"]=="user")
print("score: %d LS-Prompts, %d natuerliche Anker, %d Katalog-Prompts"%(len(texts),len(NATURAL),len(other_texts)))
nat_ppl=[prompt_ppl(t) for t in NATURAL]
nat_med=statistics.median(nat_ppl)
cat_ppl=sorted(prompt_ppl(t) for t in other_texts.values())
def cat_pct(x): return 100*sum(1 for v in cat_ppl if v<x)/len(cat_ppl)
allelo=sorted(v for v in (r.get("elo_prompt_naturalness") for r in meta) if v is not None)
med_elo=statistics.median(allelo)
def pnat(e): return 1/(1+10**((med_elo-e)/400)) if e is not None else float("nan")

print("\nnatuerliche Anker: PPL median %.1f (min %.1f, max %.1f)"%(nat_med,min(nat_ppl),max(nat_ppl)))
print("Katalog-Referenz (andere Behaviors): PPL median %.1f\n"%statistics.median(cat_ppl))
rowsout=[]
for pat,pid in rep.items():
    if pid not in texts: continue
    ppl=prompt_ppl(texts[pid]); e=elo[pat]; m=mrate[pat]
    rowsout.append((pat.split(LS+"/")[1],pid[:8],e,pnat(e),m,m*pnat(e),ppl,ppl/nat_med,cat_pct(ppl)))
print("%-26s %-9s %5s %7s %6s %8s %8s %10s %9s"%("pattern","prompt","elo","P_nat","match","K-Index","PPL","x nat.Med","Kat-Pzt"))
for r in sorted(rowsout,key=lambda r:-r[5]):
    print("%-26s %-9s %5.0f  %6.3f  %5.3f  %7.4f  %7.1f  %8.1fx  %7.0f%%"%r)
print("\nZWILLINGS-VERGLEICH (Koeder vs. natuerliche Version desselben Anliegens):")
tw={"TensorFlow":("6427c18a",NATURAL[11]),"Einkaufsliste":("645f57a9",NATURAL[12]),"10x10-Grid":("64a66df34",NATURAL[4])}
for name,(pfx,nat) in tw.items():
    k=[t for pid,t in texts.items() if pid.startswith(pfx[:8])]
    if k:
        print("  %-14s Koeder-PPL %7.1f  vs. natuerlich %6.1f  -> Faktor %.1fx"
              %(name,prompt_ppl(k[0]),prompt_ppl(nat),prompt_ppl(k[0])/prompt_ppl(nat)))
PPL_RESULTS=dict(nat_median=nat_med,rows=rowsout)
